# Processes

Section 14.2 of the notes.

A process has its own interpreter and its own memory, so processes run Python
code at the same time. The price is that starting them takes time, and that
arguments and results are converted to bytes and back.

In [ ]:
import os
import time
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np

import tasks

print("cores:", os.cpu_count())


def wall(fn):
    t0 = time.perf_counter()
    fn()
    return time.perf_counter() - t0

## 1. Speed-up on independent tasks

In [ ]:
N = os.cpu_count()

t_seq = wall(lambda: [tasks.cpu_work() for _ in range(N)])


def with_processes(k):
    with ProcessPoolExecutor(max_workers=k) as pool:
        list(pool.map(tasks.cpu_work, [tasks.CPU_SIZE] * N))


for k in (2, 4, N):
    t = wall(lambda: with_processes(k))
    print(f"{k:2d} processes {t:5.2f} s   speed-up {t_seq / t:5.2f}x")

## 2. What the pool costs before any work

Every worker starts an interpreter and imports the modules it needs.

In [ ]:
t_pool = wall(lambda: list(ProcessPoolExecutor(max_workers=N).map(int, range(N))))
print(f"starting a pool of {N} workers: {t_pool:.2f} s")
print(f"one task of tasks.cpu_work:    {wall(tasks.cpu_work):.2f} s")

A task has to be long compared with that. Many short tasks spend their time in
the transfers, which `chunksize` reduces by sending them in groups.

In [ ]:
small = [10_000] * 2_000

with ProcessPoolExecutor() as pool:
    t1 = wall(lambda: list(pool.map(tasks.cpu_work, small)))
    t2 = wall(lambda: list(pool.map(tasks.cpu_work, small, chunksize=100)))

print(f"2000 short tasks, one by one     {t1:5.2f} s")
print(f"2000 short tasks, chunksize=100  {t2:5.2f} s")

## 3. Sending data costs as well

The worker receives its arguments as bytes. Sending a large array to every task
can cost more than the computation.

In [ ]:
rng = np.random.default_rng(0)
big = rng.standard_normal(20_000_000)  # 160 MB
pieces = np.array_split(big, N)

with ProcessPoolExecutor() as pool:
    t_send = wall(lambda: list(pool.map(tasks.sum_of_copy, pieces)))

t_local = wall(lambda: [tasks.sum_of_copy(p) for p in pieces])
print(f"sending the pieces to workers {t_send:5.2f} s")
print(f"same work in this process     {t_local:5.2f} s")
print("the next notebook avoids the transfer with shared memory")

## 4. Progress while the work runs

In [ ]:
with ProcessPoolExecutor(max_workers=4) as pool:
    futures = [pool.submit(tasks.cpu_work, 2_000_000) for _ in range(8)]
    for i, f in enumerate(as_completed(futures), 1):
        f.result()
        print(f"{i}/{len(futures)} done", end="  ")